# Final Task B 17-class - MobileNetV3-Large

Train MobileNetV3-Large directly on the final 17 species classes. This uses ImageNet pretrained initialization and does not load the old 20-class Task B checkpoints.


In [1]:
from pathlib import Path
import sys

if Path('/content').exists():
    ROOT = Path('/content/woodbench')
else:
    ROOT = next((parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / 'src').exists()), Path.cwd())

sys.path.insert(0, str(ROOT))

import os
os.chdir(ROOT)
FINAL_B_DIR = ROOT / "notebooks" / "phase 1" / "task-B-17"
sys.path.insert(0, str(FINAL_B_DIR))
print(f"ROOT: {ROOT}")
print(f"task-B-17: {FINAL_B_DIR}")


ROOT: /workspace/woodbench
final-B: /workspace/woodbench/notebooks/phase 1/final-B


In [2]:
# Colab-only setup. Cloud data is expected to be already converted to 17 classes
# by 00_data_preparation.ipynb. This download only runs when the dataset is absent.
if Path('/content').exists():
    !pip install -q -r {ROOT}/requirements.txt

    DATA_DIR = ROOT / "data" / "task-B-group-split"
    if not DATA_DIR.exists():
        from huggingface_hub import login, snapshot_download
        import os

        token = os.getenv("HF_TOKEN", "")
        if token:
            login(token=token)
        snapshot_download(
            repo_id="ny0g507h3p/wood-task-B",
            repo_type="dataset",
            local_dir=str(DATA_DIR),
            ignore_patterns=["*.md", "README*"],
        )


In [3]:
from final_b_17class import (
    BACKBONE_CONFIGS,
    prepare_environment,
    run_all_experiments,
    summarize_results,
    verify_17class_dataset,
)

prepare_environment(ROOT)

BACKBONE = "mobilenetv3_large"
DISPLAY_NAME = "MobileNetV3-Large"
DATA_ROOT = ROOT / "data" / "task-B-group-split"
BATCH_SIZE = BACKBONE_CONFIGS[BACKBONE]["batch_size"]
EPOCHS = 50

print(f"Backbone: {DISPLAY_NAME} ({BACKBONE})")
print(f"Batch size: {BATCH_SIZE}")
print(f"Data root: {DATA_ROOT}")


Backbone: MobileNetV3-Large (mobilenetv3_large)
Batch size: 64
Data root: /workspace/woodbench/data/task-B-group-split


In [4]:
class_names = verify_17class_dataset(DATA_ROOT)
print(f"Classes ({len(class_names)}):")
for idx, class_name in enumerate(class_names):
    print(f"  {idx:02d}: {class_name}")


[Data] train: 17 classes, 3365 images
[Data] val: 17 classes, 903 images
[Data] test: 17 classes, 532 images
Classes (17):
  00: Araucaria araucana
  01: Callitris columellaris
  02: Calocedrus rupestris
  03: Chamaecyparis obtusa
  04: Cryptomeria japonica
  05: Cunninghamia lanceolata
  06: Fokienia hodginsii
  07: Picea glauca
  08: Pinus cernua
  09: Pinus dalatensis
  10: Pinus krempfii
  11: Pinus radiata
  12: Podocarpus pilgeri
  13: Pseudotsuga brevifolia
  14: Taxus wallichiana
  15: Thujopsis dolabrata
  16: Xanthocyparis vietnamensis


In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("CUDA not available; CPU training will be slow.")

results = run_all_experiments(
    backbone_name=BACKBONE,
    root=ROOT,
    data_root=DATA_ROOT,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    device=DEVICE,
)


GPU: NVIDIA GeForce RTX 4090
VRAM: 23.5 GB
[Data] train: 17 classes, 3365 images
[Data] val: 17 classes, 903 images
[Data] test: 17 classes, 532 images
[Device] cuda
[Output] /workspace/woodbench/data/experiments/Task-B-17class

main/seed_42/mobilenetv3_large
[main | mobilenetv3_large s42] train=3365, val=903, test=532, classes=17


model.safetensors:   0%|          | 0.00/22.1M [00:00<?, ?B/s]

Epoch 1/50 | Train Loss: 2.8544, Acc: 5.35% | Val Loss: 2.8498, Acc: 5.32% | LR: 2.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 5.32%)


Epoch 2/50 | Train Loss: 2.7520, Acc: 17.88% | Val Loss: 2.6635, Acc: 24.36% | LR: 4.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 24.36%)


Epoch 3/50 | Train Loss: 2.4495, Acc: 49.58% | Val Loss: 2.2926, Acc: 56.37% | LR: 6.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 56.37%)


Epoch 4/50 | Train Loss: 1.9224, Acc: 70.82% | Val Loss: 1.7077, Acc: 72.31% | LR: 8.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 72.31%)


... (102 outputs left out)

Epoch 6/50 | Train Loss: 0.9765, Acc: 91.20% | Val Loss: 1.0302, Acc: 86.05% | LR: 9.99e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed456_best.pth (Val Acc: 86.05%)


Epoch 7/50 | Train Loss: 0.8576, Acc: 93.39% | Val Loss: 0.9580, Acc: 88.82% | LR: 9.95e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed456_best.pth (Val Acc: 88.82%)


Epoch 8/50 | Train Loss: 0.8076, Acc: 95.34% | Val Loss: 0.9189, Acc: 90.37% | LR: 9.89e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed456_best.pth (Val Acc: 90.37%)


Epoch 9/50 [Train]:  15%|█▌        | 8/52 [00:37<02:09,  2.94s/batch, loss=0.7916, acc=95.7%]

In [ ]:
summarize_results(results)


## Continue after Vast interruption

Run the root, import/config, and data-check cells above, but do not rerun the old `run_all_experiments` cell. Then run each config cell below independently. Completed metrics are reused; interrupted checkpoints restore model, optimizer, and scheduler state.

In [5]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("CUDA not available; CPU training will be slow.")


GPU: NVIDIA GeForce RTX 4090
VRAM: 23.5 GB


In [6]:
from final_b_17class import EXPERIMENTS, load_completed_results, train_one

continued_results = {
    (row["experiment"], row["seed"]): row
    for row in load_completed_results(backbone_name=BACKBONE, root=ROOT)
}

def run_remaining(experiment_name, seed):
    result = train_one(
        root=ROOT,
        backbone=BACKBONE,
        seed=seed,
        experiment_name=experiment_name,
        experiment_cfg=EXPERIMENTS[experiment_name],
        data_root=DATA_ROOT,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        device=DEVICE,
        resume_training=True,
    )
    continued_results[(experiment_name, seed)] = result
    return result

print(f"Loaded {len(continued_results)}/6 completed configs for {BACKBONE}.")


Loaded 2/6 completed configs for mobilenetv3_large.


In [7]:
# Resume main seed 456 from its saved checkpoint
run_remaining("main", 456)


[main | mobilenetv3_large s456] train=3365, val=903, test=532, classes=17
[Resume] /workspace/woodbench/data/experiments/Task-B-17class/main/seed_456/mobilenetv3_large/mobilenetv3_large_seed456_best.pth -> epoch 10/50


Epoch 10/50 | Train Loss: 0.7548, Acc: 96.69% | Val Loss: 0.8713, Acc: 90.81% | LR: 9.70e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed456_best.pth (Val Acc: 90.81%)


Epoch 11/50 | Train Loss: 0.7450, Acc: 96.66% | Val Loss: 0.8573, Acc: 91.36% | LR: 9.57e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed456_best.pth (Val Acc: 91.36%)


Epoch 12/50 | Train Loss: 0.7271, Acc: 97.54% | Val Loss: 0.8379, Acc: 91.81% | LR: 9.41e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed456_best.pth (Val Acc: 91.81%)


Epoch 13/50 | Train Loss: 0.7086, Acc: 98.05% | Val Loss: 0.8258, Acc: 92.69% | LR: 9.24e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed456_best.pth (Val Acc: 92.69%)


Epoch 14/50 | Train Loss: 0.7066, Acc: 98.08% | Val Loss: 0.8281, Acc: 92.47% | LR: 9.05e-05[EarlyStop 1/10]


... (32 outputs left out)

Epoch 47/50 | Train Loss: 0.6428, Acc: 99.31% | Val Loss: 0.7428, Acc: 94.80% | LR: 1.09e-06[EarlyStop 9/10]


Epoch 48/50 | Train Loss: 0.6485, Acc: 99.34% | Val Loss: 0.7384, Acc: 94.91% | LR: 4.87e-07[EarlyStop 10/10]

[EarlyStopping] No improvement for 10 epochs. Stopping at epoch 48.


[main | mobilenetv3_large s456] val_acc=95.57% test_acc=98.68% F1=98.75%


{'backbone': 'mobilenetv3_large',
 'seed': 456,
 'experiment': 'main',
 'num_classes': 17,
 'class_names': ['Araucaria araucana',
  'Callitris columellaris',
  'Calocedrus rupestris',
  'Chamaecyparis obtusa',
  'Cryptomeria japonica',
  'Cunninghamia lanceolata',
  'Fokienia hodginsii',
  'Picea glauca',
  'Pinus cernua',
  'Pinus dalatensis',
  'Pinus krempfii',
  'Pinus radiata',
  'Podocarpus pilgeri',
  'Pseudotsuga brevifolia',
  'Taxus wallichiana',
  'Thujopsis dolabrata',
  'Xanthocyparis vietnamensis'],
 'best_val_acc': 95.5703211517165,
 'best_epoch': 38,
 'final_train_acc': 99.3389423076923,
 'final_val_acc': 94.90586932447397,
 'stopped_early': True,
 'epochs_trained': 48,
 'test_accuracy': 0.9868421052631579,
 'test_f1_macro': 0.9874867734593477,
 'test_precision_macro': 0.9882534606799311,
 'test_recall_macro': 0.9875251382604325,
 'test_f1_per_class': {'Araucaria araucana': 1.0,
  'Callitris columellaris': 1.0,
  'Calocedrus rupestris': 1.0,
  'Chamaecyparis obtusa': 1.

In [8]:
# Remaining 90-degree rotation ablation
run_remaining("taskB_90rotation", 42)


[taskB_90rotation | mobilenetv3_large s42] train=3365, val=903, test=532, classes=17


Epoch 1/50 | Train Loss: 2.8548, Acc: 6.10% | Val Loss: 2.8451, Acc: 5.32% | LR: 2.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 5.32%)


Epoch 2/50 | Train Loss: 2.7454, Acc: 17.28% | Val Loss: 2.6122, Acc: 34.11% | LR: 4.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 34.11%)


Epoch 3/50 | Train Loss: 2.4301, Acc: 51.83% | Val Loss: 2.1713, Acc: 67.55% | LR: 6.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 67.55%)


Epoch 4/50 | Train Loss: 1.8974, Acc: 72.66% | Val Loss: 1.5102, Acc: 80.62% | LR: 8.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 80.62%)


Epoch 5/50 | Train Loss: 1.2928, Acc: 85.16% | Val Loss: 1.0310, Acc: 87.82% | LR: 1.00e-04[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 87.82%)


... (42 outputs left out)

Epoch 48/50 | Train Loss: 0.6454, Acc: 99.28% | Val Loss: 0.6808, Acc: 97.23% | LR: 4.87e-07[EarlyStop 9/10]


Epoch 49/50 | Train Loss: 0.6449, Acc: 99.55% | Val Loss: 0.6793, Acc: 97.34% | LR: 1.22e-07[EarlyStop 10/10]

[EarlyStopping] No improvement for 10 epochs. Stopping at epoch 49.


[taskB_90rotation | mobilenetv3_large s42] val_acc=97.45% test_acc=98.50% F1=98.51%


{'backbone': 'mobilenetv3_large',
 'seed': 42,
 'experiment': 'taskB_90rotation',
 'num_classes': 17,
 'class_names': ['Araucaria araucana',
  'Callitris columellaris',
  'Calocedrus rupestris',
  'Chamaecyparis obtusa',
  'Cryptomeria japonica',
  'Cunninghamia lanceolata',
  'Fokienia hodginsii',
  'Picea glauca',
  'Pinus cernua',
  'Pinus dalatensis',
  'Pinus krempfii',
  'Pinus radiata',
  'Podocarpus pilgeri',
  'Pseudotsuga brevifolia',
  'Taxus wallichiana',
  'Thujopsis dolabrata',
  'Xanthocyparis vietnamensis'],
 'best_val_acc': 97.45293466223698,
 'best_epoch': 39,
 'final_train_acc': 99.54927884615384,
 'final_val_acc': 97.3421926910299,
 'stopped_early': True,
 'epochs_trained': 49,
 'test_accuracy': 0.9849624060150376,
 'test_f1_macro': 0.9850733679892384,
 'test_precision_macro': 0.9853108944112404,
 'test_recall_macro': 0.9855926344896934,
 'test_f1_per_class': {'Araucaria araucana': 1.0,
  'Callitris columellaris': 1.0,
  'Calocedrus rupestris': 1.0,
  'Chamaecyparis

In [9]:
# Remaining label-smoothing=0.0 ablation
run_remaining("taskB_label_smoothing_0.0", 42)


[taskB_label_smoothing_0.0 | mobilenetv3_large s42] train=3365, val=903, test=532, classes=17


Epoch 1/50 | Train Loss: 2.8546, Acc: 5.35% | Val Loss: 2.8493, Acc: 5.32% | LR: 2.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 5.32%)


Epoch 2/50 | Train Loss: 2.7405, Acc: 17.88% | Val Loss: 2.6414, Acc: 24.25% | LR: 4.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 24.25%)


Epoch 3/50 | Train Loss: 2.3999, Acc: 49.19% | Val Loss: 2.2192, Acc: 56.15% | LR: 6.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 56.15%)


Epoch 4/50 | Train Loss: 1.7841, Acc: 70.28% | Val Loss: 1.5159, Acc: 71.10% | LR: 8.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 71.10%)


Epoch 5/50 | Train Loss: 1.0086, Acc: 84.38% | Val Loss: 0.8663, Acc: 81.28% | LR: 1.00e-04[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 81.28%)


... (32 outputs left out)

Epoch 38/50 | Train Loss: 0.0323, Acc: 99.19% | Val Loss: 0.1454, Acc: 95.02% | LR: 1.65e-05[EarlyStop 9/10]


Epoch 39/50 | Train Loss: 0.0273, Acc: 99.25% | Val Loss: 0.1447, Acc: 94.35% | LR: 1.40e-05[EarlyStop 10/10]

[EarlyStopping] No improvement for 10 epochs. Stopping at epoch 39.


[taskB_label_smoothing_0.0 | mobilenetv3_large s42] val_acc=95.35% test_acc=97.93% F1=97.99%


{'backbone': 'mobilenetv3_large',
 'seed': 42,
 'experiment': 'taskB_label_smoothing_0.0',
 'num_classes': 17,
 'class_names': ['Araucaria araucana',
  'Callitris columellaris',
  'Calocedrus rupestris',
  'Chamaecyparis obtusa',
  'Cryptomeria japonica',
  'Cunninghamia lanceolata',
  'Fokienia hodginsii',
  'Picea glauca',
  'Pinus cernua',
  'Pinus dalatensis',
  'Pinus krempfii',
  'Pinus radiata',
  'Podocarpus pilgeri',
  'Pseudotsuga brevifolia',
  'Taxus wallichiana',
  'Thujopsis dolabrata',
  'Xanthocyparis vietnamensis'],
 'best_val_acc': 95.34883720930233,
 'best_epoch': 29,
 'final_train_acc': 99.24879807692308,
 'final_val_acc': 94.35215946843854,
 'stopped_early': True,
 'epochs_trained': 39,
 'test_accuracy': 0.9793233082706767,
 'test_f1_macro': 0.9799324324804509,
 'test_precision_macro': 0.9812990242167605,
 'test_recall_macro': 0.979528029160382,
 'test_f1_per_class': {'Araucaria araucana': 1.0,
  'Callitris columellaris': 1.0,
  'Calocedrus rupestris': 1.0,
  'Cham

In [10]:
# Remaining label-smoothing=0.2 ablation
run_remaining("taskB_label_smoothing_0.2", 42)


[taskB_label_smoothing_0.2 | mobilenetv3_large s42] train=3365, val=903, test=532, classes=17


Epoch 1/50 | Train Loss: 2.8542, Acc: 5.35% | Val Loss: 2.8504, Acc: 5.32% | LR: 2.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 5.32%)


Epoch 2/50 | Train Loss: 2.7634, Acc: 17.94% | Val Loss: 2.6858, Acc: 24.25% | LR: 4.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 24.25%)


Epoch 3/50 | Train Loss: 2.4990, Acc: 49.76% | Val Loss: 2.3653, Acc: 56.48% | LR: 6.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 56.48%)


Epoch 4/50 | Train Loss: 2.0576, Acc: 71.30% | Val Loss: 1.8918, Acc: 72.98% | LR: 8.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 72.98%)


Epoch 5/50 | Train Loss: 1.5906, Acc: 85.34% | Val Loss: 1.5462, Acc: 82.50% | LR: 1.00e-04[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 82.50%)


... (43 outputs left out)

Epoch 49/50 | Train Loss: 1.0788, Acc: 99.43% | Val Loss: 1.1671, Acc: 94.57% | LR: 1.22e-07[EarlyStop 6/10]


Epoch 50/50 | Train Loss: 1.0771, Acc: 99.40% | Val Loss: 1.1645, Acc: 94.91% | LR: 0.00e+00[EarlyStop 7/10]


[taskB_label_smoothing_0.2 | mobilenetv3_large s42] val_acc=95.13% test_acc=98.50% F1=98.55%


{'backbone': 'mobilenetv3_large',
 'seed': 42,
 'experiment': 'taskB_label_smoothing_0.2',
 'num_classes': 17,
 'class_names': ['Araucaria araucana',
  'Callitris columellaris',
  'Calocedrus rupestris',
  'Chamaecyparis obtusa',
  'Cryptomeria japonica',
  'Cunninghamia lanceolata',
  'Fokienia hodginsii',
  'Picea glauca',
  'Pinus cernua',
  'Pinus dalatensis',
  'Pinus krempfii',
  'Pinus radiata',
  'Podocarpus pilgeri',
  'Pseudotsuga brevifolia',
  'Taxus wallichiana',
  'Thujopsis dolabrata',
  'Xanthocyparis vietnamensis'],
 'best_val_acc': 95.12735326688815,
 'best_epoch': 43,
 'final_train_acc': 99.39903846153847,
 'final_val_acc': 94.90586932447397,
 'stopped_early': False,
 'epochs_trained': 50,
 'test_accuracy': 0.9849624060150376,
 'test_f1_macro': 0.9855333310486012,
 'test_precision_macro': 0.9866880126542564,
 'test_recall_macro': 0.9852626948215184,
 'test_f1_per_class': {'Araucaria araucana': 1.0,
  'Callitris columellaris': 1.0,
  'Calocedrus rupestris': 1.0,
  'Ch

In [11]:
results = list(continued_results.values())
summarize_results(results)



Task B 17-class summary
experiment                       seed   val_acc  test_acc        f1      prec       rec
----------------------------------------------------------------------------------------
main                               42    95.35%    98.68%    98.73%    98.82%    98.69%
main                              123    95.57%    98.50%    98.55%    98.65%    98.53%
main                              456    95.57%    98.68%    98.75%    98.83%    98.75%
taskB_90rotation                   42    97.45%    98.50%    98.51%    98.53%    98.56%
taskB_label_smoothing_0.0          42    95.35%    97.93%    97.99%    98.13%    97.95%
taskB_label_smoothing_0.2          42    95.13%    98.50%    98.55%    98.67%    98.53%
----------------------------------------------------------------------------------------
main test_acc: 98.62 +/- 0.09%
main test_f1:  98.68 +/- 0.09%
